## OBJETIVO DEL CUADERNO
En este código vamos a importar dos bases de datos, una correspondiente a la clasificación regional M49 de ONU, y otra que permite detectar cuales
son los países miembros de ONU. Vamos a descartar los países no miembros, para quedarnos con países soberanos. De esta manera evitamos sumar unidades
económicas que tienen muy baja cobertura de datos en las fuentes temporales. Además la clasificación regional propuesta por oNU (M49) presenta una
desagregación que consideramos óptima para captar fenómenos regionales.

## Importamos bases de ONU: países miembros y clasificación m49

In [1]:
import pandas as pd

df_miembros_onu = pd.read_excel("193miembros_onu.xlsx")
df_m49_onu = pd.read_excel("M49_onu.xlsx")

df_miembros_onu.head()

,Member State,M49 Code,ISO Code,Other Names,Earlier or Later Name,Earlier (a) or Later (b),Geographic Term
0,United States,840,USA,"USA, U.S.A., United States of America",NaN,NaN,UNITED STATES
1,Australia,36,AUS,Commonwealth of Australia,NaN,NaN,AUSTRALIA
2,Djibouti,262,DJI,Republic of Djibouti,French Territory of the Afars and Issas,Earlier,DJIBOUTI
3,Ghana,288,GHA,Republic of Ghana,NaN,NaN,GHANA
4,Kiribati,296,KIR,Republic of Kiribati,Gilbert Islands,Earlier,KIRIBATI


In [2]:
df_miembros_onu.columns

Index(['Member State', 'M49 Code', 'ISO Code', 'Other Names',
       'Earlier or Later Name', 'Earlier (a) or Later (b)', 'Geographic Term'],
      dtype='str')

In [3]:
df_m49_onu.columns

Index(['Region Code', 'Region Name', 'Sub-region Code', 'Sub-region Name',
       'Intermediate Region Code', 'Intermediate Region Name',
       'Country or Area', 'M49 Code', 'ISO-alpha2 Code', 'ISO-alpha3 Code',
       'Least Developed Countries (LDC)',
       'Land Locked Developing Countries (LLDC)',
       'Small Island Developing States (SIDS)'],
      dtype='str')

## Renombrar columnas

In [4]:
# Bloque 1: Renombrar columna clave en ambos df para unificar criterio de merge
df_m49_onu = df_m49_onu.rename(columns={'ISO-alpha3 Code': 'ISO-alpha3'})
df_miembros_onu = df_miembros_onu.rename(columns={'ISO Code': 'ISO-alpha3'})

## Unificamos df


In [5]:
# Bloque 2: Seleccionar únicamente las columnas de df_m49_onu que se van a incorporar
cols_m49 = [
    'ISO-alpha3',
    'Country or Area',
    'Region Code',
    'Region Name',
    'Sub-region Code',
    'Sub-region Name',
    'ISO-alpha2 Code'
]
df_m49_sub = df_m49_onu[cols_m49].copy()

In [6]:
# Bloque 3: Merge — df_miembros_onu como referencia (left join)
df_regiones_miembros_onu = df_miembros_onu.merge(
    df_m49_sub,
    on='ISO-alpha3',
    how='left',
    validate='one_to_one'  # asegura que no haya duplicados en ninguno de los dos df
)

print(df_regiones_miembros_onu.shape)
df_regiones_miembros_onu.head()

(193, 13)


,Member State,M49 Code,ISO-alpha3,Other Names,Earlier or Later Name,Earlier (a) or Later (b),Geographic Term,Country or Area,Region Code,Region Name,Sub-region Code,Sub-region Name,ISO-alpha2 Code
0,United States,840,USA,"USA, U.S.A., United States of America",NaN,NaN,UNITED STATES,United States of America,19.0,Americas,21.0,Northern America,US
1,Australia,36,AUS,Commonwealth of Australia,NaN,NaN,AUSTRALIA,Australia,9.0,Oceania,53.0,Australia and New Zealand,AU
2,Djibouti,262,DJI,Republic of Djibouti,French Territory of the Afars and Issas,Earlier,DJIBOUTI,Djibouti,2.0,Africa,202.0,Sub-Saharan Africa,DJ
3,Ghana,288,GHA,Republic of Ghana,NaN,NaN,GHANA,Ghana,2.0,Africa,202.0,Sub-Saharan Africa,GH
4,Kiribati,296,KIR,Republic of Kiribati,Gilbert Islands,Earlier,KIRIBATI,Kiribati,9.0,Oceania,57.0,Micronesia,KI


In [7]:
# Bloque 4: Formatear Region Code y Sub-region Code como categóricas, sin alterar los valores
df_regiones_miembros_onu['Region Code'] = df_regiones_miembros_onu['Region Code'].astype('Int64').astype('category')
df_regiones_miembros_onu['Sub-region Code'] = df_regiones_miembros_onu['Sub-region Code'].astype('Int64').astype('category')

df_regiones_miembros_onu.dtypes

Member State                     str
M49 Code                       int64
ISO-alpha3                       str
Other Names                      str
Earlier or Later Name            str
Earlier (a) or Later (b)         str
Geographic Term                  str
Country or Area                  str
Region Code                 category
Region Name                      str
Sub-region Code             category
Sub-region Name                  str
ISO-alpha2 Code                  str
dtype: object

In [8]:
df_regiones_miembros_onu.shape

(193, 13)

In [9]:
df_regiones_miembros_onu.head()

,Member State,M49 Code,ISO-alpha3,Other Names,Earlier or Later Name,Earlier (a) or Later (b),Geographic Term,Country or Area,Region Code,Region Name,Sub-region Code,Sub-region Name,ISO-alpha2 Code
0,United States,840,USA,"USA, U.S.A., United States of America",NaN,NaN,UNITED STATES,United States of America,19,Americas,21,Northern America,US
1,Australia,36,AUS,Commonwealth of Australia,NaN,NaN,AUSTRALIA,Australia,9,Oceania,53,Australia and New Zealand,AU
2,Djibouti,262,DJI,Republic of Djibouti,French Territory of the Afars and Issas,Earlier,DJIBOUTI,Djibouti,2,Africa,202,Sub-Saharan Africa,DJ
3,Ghana,288,GHA,Republic of Ghana,NaN,NaN,GHANA,Ghana,2,Africa,202,Sub-Saharan Africa,GH
4,Kiribati,296,KIR,Republic of Kiribati,Gilbert Islands,Earlier,KIRIBATI,Kiribati,9,Oceania,57,Micronesia,KI


## Organizar el nuevo df unificado

In [10]:
df_regiones_miembros_onu.columns

Index(['Member State', 'M49 Code', 'ISO-alpha3', 'Other Names',
       'Earlier or Later Name', 'Earlier (a) or Later (b)', 'Geographic Term',
       'Country or Area', 'Region Code', 'Region Name', 'Sub-region Code',
       'Sub-region Name', 'ISO-alpha2 Code'],
      dtype='str')

In [11]:
#Renombrar columnas 
df_regiones_miembros_onu = df_regiones_miembros_onu.rename(columns={
    'M49 Code': 'M49_country',
    'Region Code': 'M49_region',
    'Sub-region Code': 'M49_subregion'
})

In [14]:
#eliminar columnas poco útiles
df_regiones_miembros_onu = df_regiones_miembros_onu.drop(
    columns=['Earlier (a) or Later (b)', 'Earlier or Later Name', 'Geographic Term']
)

In [15]:
#Reubicar 'ISO-alpha2 Code' inmediatamente después de 'ISO-alpha3'
cols = df_regiones_miembros_onu.columns.tolist()
cols.remove('ISO-alpha2 Code')
pos = cols.index('ISO-alpha3') + 1
cols.insert(pos, 'ISO-alpha2 Code')


In [16]:
df_regiones_miembros_onu.head()

,Member State,M49_country,ISO-alpha3,Other Names,Country or Area,M49_region,Region Name,M49_subregion,Sub-region Name,ISO-alpha2 Code
0,United States,840,USA,"USA, U.S.A., United States of America",United States of America,19,Americas,21,Northern America,US
1,Australia,36,AUS,Commonwealth of Australia,Australia,9,Oceania,53,Australia and New Zealand,AU
2,Djibouti,262,DJI,Republic of Djibouti,Djibouti,2,Africa,202,Sub-Saharan Africa,DJ
3,Ghana,288,GHA,Republic of Ghana,Ghana,2,Africa,202,Sub-Saharan Africa,GH
4,Kiribati,296,KIR,Republic of Kiribati,Kiribati,9,Oceania,57,Micronesia,KI


## Exporto el df en excel

In [17]:
# Exportar df_regiones_miembros_onu a Excel
df_regiones_miembros_onu.to_excel("df_regiones_miembros_onu.xlsx", index=False)

In [18]:
df_regiones_miembros_onu.columns

Index(['Member State', 'M49_country', 'ISO-alpha3', 'Other Names',
       'Country or Area', 'M49_region', 'Region Name', 'M49_subregion',
       'Sub-region Name', 'ISO-alpha2 Code'],
      dtype='str')